# Aula 10 — Backward das ativações

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/04-deep-learning/m5-redes-neurais-do-zero/notebooks/10-backward-ativacoes-laboratorio.ipynb)

Laboratório reproduzível em **NumPy puro** para implementar e testar

$$G=H\odot\phi'(Z)$$

em sigmoid, tanh, ReLU e Leaky ReLU. O notebook publicado permanece sem outputs; execute-o em ordem para reproduzir as evidências.

## Objetivos e ambiente

- Implementar forward estável, cache e backward com shapes estritos.
- Verificar derivadas locais por diferenças centrais e teste direcional.
- Tornar explícitas as convenções da ReLU e da Leaky ReLU em zero.
- Medir saturação e atenuação multiplicativa.
- Compor uma ativação com a camada afim da Aula 09.

Dependências mínimas: Python >= 3.11, NumPy >= 1.26, Matplotlib >= 3.8 e nbformat >= 5.9 para validar o arquivo. Seed fixa: `20260910`.

In [ ]:
from importlib.metadata import version
import platform

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260910
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=9, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("nbformat:", version("nbformat"))
print("Seed:", SEED)

## 1. Forward estável e cache

Sigmoid separa entradas positivas e negativas para não avaliar exponenciais perigosas. Sigmoid e tanh guardam a saída; ReLU guarda uma máscara; Leaky ReLU guarda as inclinações. Cada cache é independente do array original.

In [ ]:
def _array(name, value):
    result = np.asarray(value, dtype=np.float64)
    if result.ndim == 0:
        raise ValueError(f"{name} deve ser tensor, não escalar")
    if not np.all(np.isfinite(result)):
        raise ValueError(f"{name} contém valor não finito")
    return result


def stable_sigmoid(z):
    z = _array("z", z)
    result = np.empty_like(z)
    positive = z >= 0
    result[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1.0 + exp_z)
    return result


def activation_forward(z, kind, alpha=0.1):
    z = _array("z", z)
    if kind == "sigmoid":
        a = stable_sigmoid(z)
        cache = (kind, a.copy(), None)
    elif kind == "tanh":
        a = np.tanh(z)
        cache = (kind, a.copy(), None)
    elif kind == "relu":
        mask = z > 0
        a = np.maximum(z, 0.0)
        cache = (kind, mask.copy(), None)
    elif kind == "leaky_relu":
        if not 0.0 < alpha < 1.0:
            raise ValueError("alpha deve pertencer a (0, 1)")
        slopes = np.where(z > 0, 1.0, alpha)
        a = np.where(z > 0, z, alpha * z)
        cache = (kind, slopes.copy(), float(alpha))
    else:
        raise ValueError(f"ativação desconhecida: {kind}")
    assert a.shape == z.shape and np.all(np.isfinite(a))
    return a, cache

## 2. Backward local

O upstream $H$ e o valor guardado no cache devem ter o mesmo shape. A multiplicação `H * local` é de Hadamard. A função não altera `H`.

In [ ]:
def activation_backward(h, cache):
    h = _array("h", h)
    kind, saved, alpha = cache
    if h.shape != saved.shape:
        raise ValueError(f"shape upstream {h.shape} incompatível com cache {saved.shape}")
    if kind == "sigmoid":
        local = saved * (1.0 - saved)
    elif kind == "tanh":
        local = 1.0 - saved**2
    elif kind == "relu":
        local = saved.astype(np.float64)
    elif kind == "leaky_relu":
        local = saved
    else:
        raise ValueError(f"cache desconhecido: {kind}")
    g = h * local
    assert g.shape == h.shape and np.all(np.isfinite(g))
    return g

## 3. Exemplo resolvido

Aplicamos as quatro funções a $Z=[-2,0,1]$ com upstream $H=[3,-4,2]$. Em zero, ReLU usa derivada 0 e Leaky ReLU usa $\alpha=0{,}1$.

In [ ]:
z_small = np.array([-2.0, 0.0, 1.0])
h_small = np.array([3.0, -4.0, 2.0])
small_results = {}

for kind in ("sigmoid", "tanh", "relu", "leaky_relu"):
    a, cache = activation_forward(z_small, kind, alpha=0.1)
    g = activation_backward(h_small, cache)
    small_results[kind] = (a, g)
    print(f"{kind:>10} | A={a} | G={g}")

assert np.allclose(small_results["relu"][1], [0.0, 0.0, 2.0])
assert np.allclose(small_results["leaky_relu"][1], [0.3, -0.4, 2.0])
assert np.allclose(small_results["sigmoid"][1], [0.314980756, -1.0, 0.393223866])
assert np.allclose(small_results["tanh"][1], [0.211952475, -4.0, 0.839948683])

## 4. Gradient checking coordenado

Para cada ativação, usamos $L(Z)=\sum H\odot\phi(Z)$. Os pontos de ReLU e Leaky ReLU ficam afastados de zero por mais que $h=10^{-6}$.

In [ ]:
def scalar_activation_loss(z, h, kind, alpha=0.1):
    a, _ = activation_forward(z, kind, alpha)
    return float(np.sum(a * h))


def central_gradient(z, evaluate, step=1e-6):
    result = np.empty_like(z, dtype=np.float64)
    for index in np.ndindex(z.shape):
        plus, minus = z.copy(), z.copy()
        plus[index] += step
        minus[index] -= step
        result[index] = (evaluate(plus) - evaluate(minus)) / (2.0 * step)
    return result


def relative_max(a, b):
    scale = np.maximum(1.0, np.maximum(np.abs(a), np.abs(b)))
    return float(np.max(np.abs(a - b) / scale))


z_check = rng.uniform(-2.5, 2.5, size=(4, 5))
z_check[np.abs(z_check) < 0.2] += np.where(z_check[np.abs(z_check) < 0.2] >= 0, 0.4, -0.4)
h_check = rng.normal(size=z_check.shape)
coordinate_errors = {}

for kind in ("sigmoid", "tanh", "relu", "leaky_relu"):
    _, cache = activation_forward(z_check, kind)
    analytic = activation_backward(h_check, cache)
    numeric = central_gradient(
        z_check, lambda value, k=kind: scalar_activation_loss(value, h_check, k)
    )
    coordinate_errors[kind] = relative_max(analytic, numeric)
    print(f"Erro relativo máximo — {kind}: {coordinate_errors[kind]:.3e}")

assert max(coordinate_errors.values()) < 2e-9

## 5. Teste direcional da VJP

Uma direção aleatória $\Delta Z$ valida todas as posições de uma só vez: $\langle G,\Delta Z\rangle$ deve coincidir com a diferença central direcional.

In [ ]:
direction = rng.normal(size=z_check.shape)
directional_errors = {}
step = 1e-6

for kind in ("sigmoid", "tanh", "relu", "leaky_relu"):
    _, cache = activation_forward(z_check, kind)
    g = activation_backward(h_check, cache)
    analytic = float(np.sum(g * direction))
    numeric = (
        scalar_activation_loss(z_check + step * direction, h_check, kind)
        - scalar_activation_loss(z_check - step * direction, h_check, kind)
    ) / (2.0 * step)
    directional_errors[kind] = abs(analytic - numeric) / max(1.0, abs(analytic), abs(numeric))
    print(f"VJP direcional — {kind}: {directional_errors[kind]:.3e}")

assert max(directional_errors.values()) < 2e-9

## 6. A quina não é um gradient check comum

Em zero, a diferença central da ReLU atravessa os dois ramos e retorna 0,5. Nossa convenção de backward retorna 0. Para Leaky ReLU com $\alpha=0{,}1$, a diferença central retorna $(1+\alpha)/2=0{,}55$, enquanto a convenção retorna $\alpha$.

In [ ]:
step = 1e-6
relu_numeric_zero = (
    scalar_activation_loss(np.array([step]), np.ones(1), "relu")
    - scalar_activation_loss(np.array([-step]), np.ones(1), "relu")
) / (2.0 * step)
_, relu_zero_cache = activation_forward(np.zeros(1), "relu")
relu_convention = activation_backward(np.ones(1), relu_zero_cache)[0]

leaky_numeric_zero = (
    scalar_activation_loss(np.array([step]), np.ones(1), "leaky_relu")
    - scalar_activation_loss(np.array([-step]), np.ones(1), "leaky_relu")
) / (2.0 * step)
_, leaky_zero_cache = activation_forward(np.zeros(1), "leaky_relu")
leaky_convention = activation_backward(np.ones(1), leaky_zero_cache)[0]

assert relu_convention == 0.0 and np.isclose(relu_numeric_zero, 0.5)
assert np.isclose(leaky_convention, 0.1) and np.isclose(leaky_numeric_zero, 0.55)
print(f"ReLU em zero: convenção={relu_convention:.2f}; central={relu_numeric_zero:.2f}")
print(f"Leaky em zero: convenção={leaky_convention:.2f}; central={leaky_numeric_zero:.2f}")

## 7. Saturação em precisão finita

Calculamos derivadas locais em pré-ativações de -1000 a 1000. O forward estável não emite overflow e todos os resultados permanecem finitos.

In [ ]:
z_extreme = np.array([-1000.0, -20.0, -5.0, 0.0, 5.0, 20.0, 1000.0])
local_derivatives = {}

for kind in ("sigmoid", "tanh", "relu", "leaky_relu"):
    _, cache = activation_forward(z_extreme, kind)
    local_derivatives[kind] = activation_backward(np.ones_like(z_extreme), cache)
    assert np.all(np.isfinite(local_derivatives[kind]))

assert np.max(local_derivatives["sigmoid"]) == 0.25
assert np.max(local_derivatives["tanh"]) == 1.0
assert local_derivatives["sigmoid"][0] == 0.0
assert local_derivatives["tanh"][-1] == 0.0

print("z:", z_extreme)
for kind, derivative in local_derivatives.items():
    print(f"{kind:>10}:", derivative)

## 8. Produto de derivadas ao longo de um caminho

Este experimento não treina uma rede. Ele isola o efeito multiplicativo de uma derivada local constante durante 1 a 50 etapas.

In [ ]:
depth = np.arange(1, 51)
factors = {
    "sigmoid no centro (0,25)": 0.25,
    "tanh em z=2": float(1.0 - np.tanh(2.0) ** 2),
    "ReLU ativa (1)": 1.0,
    "Leaky negativa (0,1)": 0.1,
}
chains = {name: factor**depth for name, factor in factors.items()}

sigmoid_depth_20 = chains["sigmoid no centro (0,25)"][19]
assert np.isclose(sigmoid_depth_20, 0.25**20)
assert np.isclose(sigmoid_depth_20, 9.094947017729282e-13)
assert np.all(chains["ReLU ativa (1)"] == 1.0)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for name, values in chains.items():
    ax.plot(depth, values, label=name)
ax.set_yscale("log")
ax.set_xlabel("Quantidade de derivadas multiplicadas")
ax.set_ylabel("Magnitude do produto")
ax.set_title("Atenuação ao longo de um caminho")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Sigmoid, profundidade 20: {sigmoid_depth_20:.12e}")
print("Texto alternativo: curvas logarítmicas mostram produtos de derivadas locais entre 1 e 50 etapas.")

## 9. Localidade, shape e upstream imutável

Alterar uma posição do upstream deve alterar apenas a mesma posição do gradiente. Também confirmamos que o backward não modifica seu argumento.

In [ ]:
z_local = rng.normal(size=(3, 4))
h_base = rng.normal(size=z_local.shape)
h_before = h_base.copy()
_, cache_local = activation_forward(z_local, "tanh")
g_base = activation_backward(h_base, cache_local)

h_changed = h_base.copy()
h_changed[1, 2] += 7.0
g_changed = activation_backward(h_changed, cache_local)
changed_positions = np.argwhere(np.abs(g_changed - g_base) > 1e-12)

assert np.array_equal(h_base, h_before)
assert changed_positions.shape == (1, 2)
assert np.array_equal(changed_positions[0], [1, 2])
assert g_base.shape == z_local.shape == h_base.shape
print("Posição afetada:", changed_positions.tolist())
print("Upstream preservado:", np.array_equal(h_base, h_before))

## 10. Contratos falham cedo

O laboratório rejeita shape incompatível, escalar, ativação desconhecida e $\alpha$ inválido antes de propagar um erro silencioso.

In [ ]:
messages = []
tests = [
    lambda: activation_backward(np.ones((2, 2)), cache_local),
    lambda: activation_forward(1.0, "tanh"),
    lambda: activation_forward(np.ones(2), "desconhecida"),
    lambda: activation_forward(np.ones(2), "leaky_relu", alpha=1.0),
]
for test in tests:
    try:
        test()
    except ValueError as exc:
        messages.append(str(exc))
    else:
        raise AssertionError("contrato inválido deveria falhar")

assert len(messages) == 4
for message in messages:
    print("Contrato acionado:", message)

## 11. Composição afim + tanh

Para $A=\tanh(XW+b)$ e $L=\langle A,H\rangle_F$, calculamos primeiro $G=H\odot(1-A^2)$ e depois os três gradientes afins. Diferenças centrais validam a composição, não apenas cada peça isolada.

In [ ]:
def affine_activation_loss(x, w, b, h):
    z = x @ w + b
    a, _ = activation_forward(z, "tanh")
    return float(np.sum(a * h))


x = rng.normal(size=(4, 3))
w = rng.normal(size=(3, 2))
b = rng.normal(size=2)
h = rng.normal(size=(4, 2))
z = x @ w + b
a, activation_cache = activation_forward(z, "tanh")
g = activation_backward(h, activation_cache)
dx = g @ w.T
dw = x.T @ g
db = g.sum(axis=0)

num_dx = central_gradient(x, lambda value: affine_activation_loss(value, w, b, h))
num_dw = central_gradient(w, lambda value: affine_activation_loss(x, value, b, h))
num_db = central_gradient(b, lambda value: affine_activation_loss(x, w, value, h))
composition_errors = {
    "X": relative_max(dx, num_dx),
    "W": relative_max(dw, num_dw),
    "b": relative_max(db, num_db),
}

assert dx.shape == x.shape and dw.shape == w.shape and db.shape == b.shape
assert max(composition_errors.values()) < 2e-9
for name, error in composition_errors.items():
    print(f"Erro composto em {name}: {error:.3e}")

## 12. Visualização das funções e derivadas

O painel permite comparar faixa de saída, saturação e quinas. Linhas verticais marcam zero; as convenções em zero são valores discretos da implementação, não derivadas clássicas.

In [ ]:
grid = np.linspace(-6.0, 6.0, 1201)
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)

for ax, kind in zip(axes.ravel(), ("sigmoid", "tanh", "relu", "leaky_relu")):
    output, cache = activation_forward(grid, kind)
    derivative = activation_backward(np.ones_like(grid), cache)
    ax.plot(grid, output, label="ativação")
    ax.plot(grid, derivative, label="derivada local")
    ax.axvline(0.0, color="black", linewidth=0.7, alpha=0.5)
    ax.set_title(kind)
    ax.grid(alpha=0.2)
    ax.legend()

fig.suptitle("Ativações e derivadas locais")
plt.tight_layout()
plt.show()
assert all(len(ax.lines) >= 3 for ax in axes.ravel())
print("Texto alternativo: quatro painéis com sigmoid, tanh, ReLU e Leaky ReLU e suas derivadas entre -6 e 6.")

## 13. Auditoria final

Os grupos abaixo consolidam exemplo manual, derivadas coordenadas e direcionais, convenções, estabilidade, localidade, contratos e composição.

In [ ]:
checks = {
    "ReLU manual": np.allclose(small_results["relu"][1], [0, 0, 2]),
    "Leaky manual": np.allclose(small_results["leaky_relu"][1], [0.3, -0.4, 2]),
    "gradient checks locais": max(coordinate_errors.values()) < 2e-9,
    "VJPs direcionais": max(directional_errors.values()) < 2e-9,
    "convenção ReLU": relu_convention == 0.0,
    "convenção Leaky": np.isclose(leaky_convention, 0.1),
    "extremos finitos": all(np.all(np.isfinite(v)) for v in local_derivatives.values()),
    "sigmoid limitada": np.all((small_results["sigmoid"][0] >= 0) & (small_results["sigmoid"][0] <= 1)),
    "tanh limitada": np.all(np.abs(small_results["tanh"][0]) <= 1),
    "atenuação confirmada": np.isclose(sigmoid_depth_20, 0.25**20),
    "upstream imutável": np.array_equal(h_base, h_before),
    "localidade": np.array_equal(changed_positions, [[1, 2]]),
    "contratos": len(messages) == 4,
    "composição afim+tanh": max(composition_errors.values()) < 2e-9,
}
assert all(checks.values())
for name, passed in checks.items():
    print(f"[{'OK' if passed else 'FALHOU'}] {name}")
print(f"{len(checks)} grupos de verificações concluídos.")

## Conclusões

- O backward das quatro ativações preservou shapes e o upstream.
- Diferenças centrais validaram pontos suaves; a quina foi tratada como convenção separada.
- Sigmoid e tanh saturaram nas caudas, enquanto ReLU transmitiu ou bloqueou o gradiente por máscara.
- A composição afim + tanh recuperou gradientes de $X$, $W$ e $b$ numericamente.

Na próxima aula, derivaremos o backward das losses desde o escalar até predições e logits, sem saltos algébricos.